In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 2.8 The Brachistochrone and Tautochrone

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume II — Analytical Mechanics",
    number="2.8",
    title="The Brachistochrone and Tautochrone",
    blurb="The curve of fastest descent and the curve of equal-time descent — "
    "both the same cycloid. The founding problem of the calculus of "
    "variations, solved and animated.",
    difficulty="advanced",
    estimate="85–110 min",
)

## Notebook overview

In 1696 Johann Bernoulli challenged the mathematicians of Europe: a bead slides
without friction from a point $A$ to a lower point $B$ under gravity: along
*which curve* does it arrive **soonest**? Not the straight line. The answer is a
**cycloid**, and finding it launched the **calculus of variations**: the
mathematics of optimising over an entire *function*, not a number. That same
machinery, the Euler–Lagrange equation applied to a functional, is exactly what
underlies Hamilton's principle and the Lagrangian mechanics of [§2.1](lagrangian-sympy.ipynb): the
path a system takes extremises an action. So this classic problem returns us to
the mathematical root of Volume II's mechanics.

Two miracles live on this one curve. The **brachistochrone** ("shortest time"):
the cycloid is the fastest descent. The **tautochrone** ("same time"): a bead
released from rest anywhere on a cycloid reaches the bottom in *exactly the same
time*, independent of where it started: Huygens' isochronous pendulum. We will
derive the cycloid variationally, confirm it beats its rivals, prove the
equal-time property, and animate both: the synchronized beads and the race.

> **How to read the checks.** Each exercise ends with a `validate` call against
> an independent fact: a closed-form descent time, a symbolic first integral, an
> equal-time prediction. A ✓ is strong evidence we got it right; a ✗ is a prompt
> to *locate the discrepancy*, not a verdict.

> **Scope.** A working review, not a course in the calculus of variations: we
> develop just enough, here and in the problem statements. See Goldstein,
> *Classical Mechanics*, ch. 2 {cite}`goldstein` and Nolting,
> *Theoretische Physik 2* {cite}`nolting2`; the historical source is Johann
> Bernoulli's 1696 challenge.

## Theory in brief — a calculus-of-variations primer

### Functionals and the variational problem

A **functional** maps a whole *function* to a number. The bead's descent time is
one: with the curve written $y(x)$ from $A=(0,0)$ down to $B$, energy
conservation from rest gives the speed $v=\sqrt{2g(y_0-y)}$ at height $y$ (here
$y_0=0$), and the time is the integral of $ds/v$ along the path,

```{math}
:label: eq-functional
T[y] = \int_A^B \frac{ds}{v}
     = \int_0^{x_B} \frac{\sqrt{1+y'^2}}{\sqrt{2g\,(y_0-y)}}\;dx .
```

We seek the *function* $y(x)$ that makes $T[y]$ **stationary** (a minimum): a
problem in the calculus of variations, not ordinary calculus.

### The Euler–Lagrange equation for a functional

For any functional $J[y]=\int F(y,y',x)\,dx$, the stationary function satisfies
the **Euler–Lagrange equation**

```{math}
:label: eq-el-functional
\frac{d}{dx}\!\left(\frac{\partial F}{\partial y'}\right) - \frac{\partial F}{\partial y} = 0 .
```

This is *identical in form* to the equation of motion in Lagrangian mechanics
([§2.1](lagrangian-sympy.ipynb)): there the independent variable is time and $F$ is the Lagrangian
$L=T-V$; here it is the coordinate $x$ and $F$ is the time-integrand. The
Lagrangian formalism **is** a variational principle (Hamilton's principle).

### The Beltrami identity

When $F$ has **no explicit $x$-dependence** (as here, $x$ appears only through
$y$ and $y'$), {eq}`eq-el-functional` has a first integral, the **Beltrami
identity**:

```{math}
:label: eq-beltrami
F - y'\,\frac{\partial F}{\partial y'} = \text{const}.
```

One line proves it: differentiate the left-hand side in $x$ and substitute
{eq}`eq-el-functional`; every term cancels (Goldstein, *Classical Mechanics*,
ch. 2 {cite}`goldstein`, treats these first integrals in full).
This is the variational cousin of energy conservation (a cyclic variable), and it
turns the brachistochrone's second-order problem into a first-order one we can
solve.

### The solution is a cycloid

Feeding $F=\sqrt{1+y'^2}/\sqrt{-2gy}$ into {eq}`eq-beltrami` collapses (Exercise
2) to $y\,(1+y'^2)=\text{const}$, whose solution is the **cycloid**: the curve
traced by a point on a circle of radius $a$ rolling along the ceiling:

```{math}
:label: eq-cycloid
x = a\,(\varphi - \sin\varphi), \qquad y = -a\,(1 - \cos\varphi),
```

with $\varphi$ the rolling angle. Its cusp is at $A$; it sweeps to the low point
at $\varphi=\pi$.

### The tautochrone property

On the cycloid the descent time to the bottom is, astonishingly,

```{math}
:label: eq-tautochrone
T_\text{bottom} = \pi\sqrt{\tfrac{a}{g}}, \qquad\text{independent of the start height,}
```

because motion along the arc is *simple harmonic* in arclength (Exercises 5–6).
Release a bead from anywhere on the curve and it reaches the bottom in the same
time: Huygens' tautochrone. The brachistochrone and the tautochrone are the
**same curve**.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp, quad
from scipy.interpolate import interp1d
import sympy as sp

from ecp import draw, validate
from ecp.animate import show

A_RAD = 1.0  # cycloid radius a  [m]
G = 9.81  # gravitational acceleration  [m/s²]
# Endpoints of the brachistochrone: the cusp A=(0,0) to the low point of one arch.
X_B, Y_B = np.pi * A_RAD, -2.0 * A_RAD  # B = (πa, -2a)


def cycloid(phi, a=A_RAD):
    """Cartesian points of the brachistochrone cycloid (eq-cycloid).

    The curve a rolling-circle point traces, the shape that minimises descent time.

    Parameters
    ----------
    phi : numpy.ndarray
        Cycloid parameter (rolling angle).
    a : float, optional
        Rolling-circle radius (default the module constant).

    Returns
    -------
    x, y : numpy.ndarray
        The cycloid coordinates.
    """
    return a * (phi - np.sin(phi)), -a * (1.0 - np.cos(phi))


def path_time(xs, ys, g=G):
    """Frictionless descent time from rest along a sampled path.

    A midpoint-rule integral of ds/v with v = sqrt(2g·(y0 − y)), for comparing
    competing tracks.

    Parameters
    ----------
    xs, ys : numpy.ndarray
        Sampled path coordinates (descending).
    g : float, optional
        Gravitational acceleration.

    Returns
    -------
    float
        The total descent time, in seconds.
    """
    xs, ys = np.asarray(xs, float), np.asarray(ys, float)
    y0 = ys[0]
    ds = np.hypot(np.diff(xs), np.diff(ys))
    y_mid = 0.5 * (ys[:-1] + ys[1:])
    v = np.sqrt(np.maximum(2 * g * (y0 - y_mid), 1e-30))
    return float(np.sum(ds / v))


def descent_time_func(y_func, yp_func, x_end, g=G, y0=0.0):
    """Descent time for an explicit curve y(x) from the functional.

    Evaluates the descent-time integral (eq-functional in the theory section)
    by quadrature, with a substitution x = u^2 that tames the 1/sqrt(x)
    start-from-rest singularity.

    Parameters
    ----------
    y_func : callable
        The height $y(x)$.
    yp_func : callable
        Its derivative $y'(x)$.
    x_end : float
        Right endpoint.
    g : float, optional
        Gravity.
    y0 : float, optional
        Starting height (default 0).

    Returns
    -------
    float
        The descent time, in seconds.
    """

    def integrand(u):
        x = u * u
        denom = np.sqrt(max(2 * g * (y0 - y_func(x)), 1e-300))
        return np.sqrt(1 + yp_func(x) ** 2) / denom * 2 * u

    val, _ = quad(integrand, 0.0, np.sqrt(x_end), limit=200)
    return val


# Cycloid arclength measured from the bottom (φ=π): s = 4a·cos(φ/2); motion in
# s is simple harmonic with ω = (1/2)·sqrt(g/a): the root of the tautochrone
# property.
OMEGA = 0.5 * np.sqrt(G / A_RAD)


def descent_time_ode(phi0, a=A_RAD, g=G):
    """Descent time to the cycloid's bottom by integrating the bead motion.

    Integrates the cycloid bead's equation of motion from release at angle
    phi0 — the numerical side of the tautochrone check.

    Parameters
    ----------
    phi0 : float
        Release angle on the cycloid.
    a : float, optional
        Rolling-circle radius.
    g : float, optional
        Gravity.

    Returns
    -------
    float
        The descent time, in seconds.
    """
    s0 = 4 * a * np.cos(phi0 / 2)

    def rhs(t, st):
        return [st[1], -(OMEGA**2) * st[0]]

    def hit_bottom(t, st):
        return st[0]

    hit_bottom.terminal = True
    hit_bottom.direction = -1.0
    sol = solve_ivp(
        rhs, (0.0, 10.0), [s0, 0.0], events=hit_bottom, rtol=1e-11, atol=1e-12
    )
    return float(sol.t_events[0][0])

## Exercise 1 — The descent-time functional

The whole problem rests on {eq}`eq-functional`: a
frictionless bead released from rest at $A$ has, by energy conservation, speed
$v=\sqrt{2g(y_0-y)}$ wherever it has dropped to height $y$; the time to traverse
a curve is $T=\int ds/v$, a *functional* of the curve. Before optimising it, we
anchor it on the one curve everyone reaches for first (the **straight line**)
for which the motion is uniform acceleration down an incline and the time has the
closed form $T_\text{line}=\sqrt{2(x_B^2+y_B^2)/(g\,|y_B|)}$.

**This exercise.**

1. Implement `path_time` for a sampled curve (given in Setup): a
   midpoint-rule integral of $ds/v$.
2. Compute the straight-line descent time from $A=(0,0)$ to $B=(\pi a,-2a)$
   numerically and against the closed form.

The setup is {numref}`fig-brach-setup`.

In [ ]:
# (solution hidden on the public site)


### Solution — Exercise 1

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    T_line_numeric,
    T_line_closed,
    "straight-line descent time matches the closed form",
    rtol=1e-3,
)

## Exercise 2 — The variational derivation of the cycloid

Apply the Euler–Lagrange machinery to
{eq}`eq-functional`. Because the integrand $F=\sqrt{1+y'^2}/\sqrt{-2gy}$ carries
*no explicit $x$*, the Beltrami identity {eq}`eq-beltrami` applies. Computing
$F-y'\,\partial F/\partial y'$ and simplifying gives the compact first integral

$$
y\,(1 + y'^2) = -2a = \text{const},
$$

a separable first-order ODE. Its solution, parametrised by the rolling angle
$\varphi$, is exactly the cycloid {eq}`eq-cycloid`. Rather than grind the
separation by hand, we **verify** the cycloid against the first integral
symbolically with SymPy: the same hand-vs-symbolic discipline as [§2.1](lagrangian-sympy.ipynb).

**This exercise.**

1. Form $y'(\varphi)=\dot y/\dot x$ for the cycloid (`sympy.diff`).
2. Show $y(1+y'^2)$ simplifies to the constant $-2a$ (`sympy.simplify`), i.e.
   the cycloid satisfies the brachistochrone first integral derived from
   {eq}`eq-beltrami`.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    bool(beltrami_holds),
    "the cycloid solves the brachistochrone first integral y(1+y'²)=const",
    f"y(1+y'²) = {first_integral}",
)

## Exercise 3 — The cycloid beats the competition

The variational argument says the cycloid is
*stationary*; here we confirm it is the *minimum* by racing it, as a descent
time, against two honest rivals between the **same** endpoints: the straight line
and a curve that bows below the chord (a tuned sag). One caution sets up the
computation: $v=\sqrt{-2gy}$ is real only while $y\le y_0=0$, so any rival must
stay at or below the start height: a curve that rises above $A$ would make the
speed imaginary. Our sag $y(x)=-\tfrac{2}{\pi}x - d\sin x$ (with $d>0$) dips below
the chord yet never rises above $A$.

**This exercise.** Compute the descent time along the cycloid (closed form
$\pi\sqrt{a/g}$, Exercise 4), the straight line, and the sag, and show the
cycloid is fastest ({numref}`fig-brach-curves`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    T_cycloid < T_line and T_cycloid < T_sag,
    "the cycloid is the curve of fastest descent",
    f"cycloid {T_cycloid:.3f} < line {T_line:.3f}, sag {T_sag:.3f}",
)

## Exercise 4 — Descent time along the cycloid, in closed form

On the cycloid the descent time has a clean closed
form. Measuring arclength $s$ from the low point, one finds $s=4a\cos(\varphi/2)$
and that the height above the bottom is $s^2/(8a)$, so a bead on the cycloid
obeys $\ddot s = -\tfrac{g}{4a}\,s$, **simple harmonic motion** with
$\omega=\tfrac12\sqrt{g/a}$. A bead released from the cusp ($\varphi=0$) reaches
the bottom in a quarter period, $T=\pi\sqrt{a/g}$: the result {eq}`eq-tautochrone`
(and reaching angle $\varphi$ takes $\varphi\sqrt{a/g}$).

**This exercise.** Integrate the arclength dynamics from rest at the cusp with
`descent_time_ode` (a `scipy.integrate.solve_ivp` solve stopped by a bottom-arrival
event) and confirm the descent-to-bottom time equals $\pi\sqrt{a/g}$.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    T_cycloid_numeric,
    np.pi * np.sqrt(A_RAD / G),
    "cycloid descent-to-bottom time is π√(a/g)",
    rtol=1e-6,
)

## Exercise 5 — The tautochrone property

Here is the headline. Because the arclength motion
is simple harmonic with a frequency $\omega=\tfrac12\sqrt{g/a}$ that **does not
depend on amplitude**, the time to reach the bottom is a quarter period
*whatever the release point*: the tautochrone property {eq}`eq-tautochrone`.
Release one bead from just above the bottom and another from the cusp: they
arrive together.

**This exercise.** Compute the descent-to-bottom time for several release angles
$\varphi_0$ and confirm they are all equal to $\pi\sqrt{a/g}$.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    times_from_phi0,
    T_expected * np.ones_like(times_from_phi0),
    "every start point reaches the bottom in the same time (tautochrone)",
    rtol=1e-6,
)

## Exercise 6 — Synchronized beads (worked animation)

Motion is the whole point of the tautochrone, so this is genuinely an animation.
Place several beads at *different* heights on the same cycloid, release them all
from rest at the same instant, and watch them slide down: they reach the lowest
point **in unison** despite the different starts. The arclength SHM gives each
bead's position exactly: $\varphi(t)=2\arccos\!\big(\cos(\varphi_0/2)\cos\omega t\big)$.
The validation checks the **physics of the animated data** (that the arrival
times agree), not the drawing.

In [ ]:
# (solution hidden on the public site)


### Validation 6

Integrating each bead's dynamics independently, all arrival times must agree.

In [ ]:
arrival_times = np.array([descent_time_ode(p) for p in bead_phi0])
validate.close(
    arrival_times,
    arrival_times[0] * np.ones_like(arrival_times),
    "all beads arrive at the bottom simultaneously",
    rtol=1e-3,
)

## Exercise 7 — The brachistochrone race (student animation)

Now a *race*, where motion is again the point: release three beads at the same
instant from $A$, each constrained to a different track (cycloid, straight line,
sag) all ending at $B$. The cycloid bead wins, **even though its path is
longer**, because it plunges early to build speed. Here you build the animation.

The dynamics along an arbitrary track follow from $\ddot s = -g\,\frac{dy}{ds}$
(the tangential component of gravity), which `race_curve` integrates with
`scipy.integrate.solve_ivp` for the line and sag; the cycloid bead uses its
exact SHM solution.

1. Obtain each bead's position over time (`race_curve` for the line and sag;
   the closed-form SHM for the cycloid).
2. **Animate the three beads racing**, then `plt.close(fig)` and end with
   `ecp.animate.show(anim)`.
3. Confirm the cycloid bead arrives first by comparing the three arrival
   times.

A ✗ points at the descent integration or arrival times, not the animation.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    arrival_cyc < arrival_line and arrival_cyc < arrival_sag,
    "the cycloid bead wins the race",
    f"cycloid {arrival_cyc:.3f} s < line {arrival_line:.3f} s, sag {arrival_sag:.3f} s",
)

## Notebook summary

- The descent-time **functional** $T[y]=\int\sqrt{(1+y'^2)/(-2gy)}\,dx$ and its variational
  minimisation to the **cycloid**, confirmed to beat the straight line and a
  curve bowed below the chord.
- The closed-form descent time along the cycloid; the **tautochrone** property (equal descent
  time from any starting height), shown with synchronized beads; and a brachistochrone race.

## Outlook

- **Snell's law / Fermat's principle.** Johann Bernoulli's own 1696 solution
  recast the bead as a light ray refracting through ever-faster media: the
  brachistochrone as an optics problem, and a first glimpse of the deep tie
  between least-time and least-action.
- **Huygens' isochronous pendulum.** A pendulum forced onto a cycloidal path is
  *exactly* isochronous: its period is independent of amplitude, unlike the
  circular pendulum of [§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb) whose period grows with swing. The tautochrone is why.
- **Constrained variation.** Isoperimetric problems (the catenary, the hanging
  chain from the "going further" of [§1.7](../01-elementary-mechanics/falling-chain.ipynb), and the shape of a soap film) add a constraint
  via a Lagrange multiplier.
- **Geodesics** as variational problems (the shortest path on a curved surface)
  point forward to general relativity.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()